In [0]:
from pyspark.sql import functions as F

In [0]:
#Constants
SOURCE_TABLE_NAME = "customers"
TABLE_NAME = "customers_raw"
SCHEMA = "bronze"
INGEST_CATALOG = "sl_ingest"

In [0]:
#Configs
configs = dict(dbutils.notebook.entry_point.getCurrentBindings())
ENV = configs.get('env', 'dev')
INITIAL_RUN = configs.get('initial_run', False)

CATALOG        = f"sl_{ENV}"
CHECKPOINT_BASE = f"/Volumes/{CATALOG}/{SCHEMA}/checkpoints"

print(f"ENV={ENV} | source schema: {INGEST_CATALOG}.{ENV} | catalog: {CATALOG} | checkpoints: {CHECKPOINT_BASE}")

In [0]:
source_table = f"{INGEST_CATALOG}.{ENV}.{SOURCE_TABLE_NAME}"
target_table = f"{CATALOG}.{SCHEMA}.{TABLE_NAME}"
checkpoint_path = f"{CHECKPOINT_BASE}/{TABLE_NAME}/"
print(source_table, target_table, checkpoint_path)

In [ ]:
if INITIAL_RUN: #this is not an ideal production pattern but more straight forward for this used case
    df = spark.read.table(source_table)
    (df
     .withColumn("_ingested_at", F.current_timestamp())
     .writeTo(target_table)
     .using("delta")
     .create()) #throws an error if run accidently after table creation
    
    spark.sql(f"ALTER TABLE {target_table} SET TBLPROPERTIES ('delta.deletedFileRetentionDuration' = 'interval 90 days')")
    #longer file retention due to cdc properties
    spark.sql(f"ALTER TABLE {target_table} SET TBLPROPERTIES ('delta.logRetentionDuration' = 'interval 90 days')")
    spark.sql(f"ALTER TABLE {target_table} CLUSTER BY (_ingested_at)")
    
    dbutils.notebook.exit(f"Initial Table: {source_table} Setup Finished Successfully")

In [0]:
df = (
    spark.readStream
    .option("readChangeFeed", "true")
    .table(source_table)
    .withColumn("_ingested_at", F.current_timestamp())
    )

In [0]:
query = (df.writeStream
    .trigger(availableNow=True)
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .option("mergeSchema", "true")
    .outputMode("append")
    .toTable(target_table))

query.awaitTermination()